# Notebook 01 — Exploratory Data Analysis & Dataset

This notebook validates the quality and structure of the supply-chain-detector training dataset
before it enters the feature engineering and model training pipeline.

**Dataset composition:**
- **Malicious packages** — sourced from the [Backstabbers Knife Collection](https://github.com/IQTLabs/Backstabbers-Knife-Collection) (known supply-chain attacks)
- **Benign packages** — top-1000 packages from PyPI and npm registries

**Questions answered here:**
1. How balanced are the classes?
2. How are packages distributed across registries?
3. Which fields are missing?
4. Do package names differ structurally between benign and malicious?
5. How much source code is available for downstream analysis?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Imports ready.")

In [ ]:
with open("../data/processed/malicious_normalized.json", "r", encoding="utf-8") as f:
    mal_data = json.load(f)

with open("../data/processed/benign_normalized.json", "r", encoding="utf-8") as f:
    ben_data = json.load(f)

# Convert lists of dicts → pandas DataFrames
df_mal = pd.DataFrame(mal_data)
df_ben = pd.DataFrame(ben_data)

# Combine into one DataFrame with all records
df_all = pd.concat([df_mal, df_ben], ignore_index=True)

print(f"Malicious: {len(df_mal)} records")
print(f"Benign:    {len(df_ben)} records")
print(f"Total:     {len(df_all)} records")

## 1. Label Distribution
How many malicious vs benign records do we have?

In [ ]:
label_counts = df_all["label"].value_counts()

print(label_counts)
print(f"\nRatio (benign:malicious): {label_counts.get('benign', 0)}:{label_counts.get('malicious', 0)}")

# Plot a bar chart
label_counts.plot(kind="bar", color=["#2ecc71", "#e74c3c"])
plt.title("Label Distribution")
plt.ylabel("Count")
plt.show()


## 2. Registry Distribution
How are records distributed across PyPI and npm?


In [ ]:
# Cross-tabulate: label × registry
cross_tab = pd.crosstab(df_all["label"], df_all["registry"])
print(cross_tab)

# Plot grouped bar chart
cross_tab.plot(kind="bar")
plt.title("Registry Distribution by Label")
plt.ylabel("Count")
plt.show()


## 3. Missing Field Rates
Which fields have missing (null/None) values?


In [ ]:
# Calculate missing rate for each column
missing = df_all.isnull().sum()
missing_pct = (missing / len(df_all) * 100).round(2)

missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_report = missing_report.sort_values(by="missing_count", ascending=False)
print(missing_report)


## 4. Sample Records
Quick look at the actual data:


In [ ]:
print("=== Malicious samples ===")
df_mal.head(3)


print("=== Benign samples ===")
df_ben.head(3)


## 5. Package Name Analysis

Do malicious packages have different naming patterns than benign ones?
Typosquatting attacks typically produce names that are 1–2 characters away from popular packages.

In [ ]:
# Package name length distribution
df_mal["name_len"] = df_mal["package_name"].str.len()
df_ben["name_len"] = df_ben["package_name"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(df_ben["name_len"], bins=30, alpha=0.6, color="#2ecc71", label="Benign", edgecolor="white")
ax.hist(df_mal["name_len"], bins=15, alpha=0.7, color="#e74c3c", label="Malicious", edgecolor="white")
ax.set_xlabel("Package Name Length")
ax.set_ylabel("Count")
ax.set_title("Name Length Distribution", fontweight="bold")
ax.legend()

# Box plot (pure matplotlib)
ax = axes[1]
bp = ax.boxplot(
    [df_ben["name_len"].dropna(), df_mal["name_len"].dropna()],
    tick_labels=["Benign", "Malicious"],
    patch_artist=True,
    widths=0.5,
)
bp["boxes"][0].set_facecolor("#2ecc71")
bp["boxes"][1].set_facecolor("#e74c3c")
for b in bp["boxes"]:
    b.set_alpha(0.6)
ax.set_title("Name Length by Label", fontweight="bold")
ax.set_ylabel("Characters")

plt.tight_layout()
plt.show()

print(f"Benign    — mean: {df_ben['name_len'].mean():.1f}, median: {df_ben['name_len'].median():.0f}")
print(f"Malicious — mean: {df_mal['name_len'].mean():.1f}, median: {df_mal['name_len'].median():.0f}")

## 6. Source Code Availability

The detection pipeline has five layers. Layers 2 (embeddings) and 3 (static analysis) require
source code. How much of our dataset has code available for these layers?

> **Note:** In the live API pipeline, source code is extracted from package tarballs on-demand.
> This offline dataset primarily contains metadata.

In [ ]:
# Source code availability
df_all["has_source"] = df_all["source_code"].apply(
    lambda x: bool(x and str(x).strip()) if pd.notna(x) else False
)

source_stats = df_all.groupby("label")["has_source"].agg(["sum", "count"])
source_stats["pct"] = (source_stats["sum"] / source_stats["count"] * 100).round(1)
source_stats.columns = ["with_source", "total", "pct_with_source"]

print("Source Code Availability by Label:")
print(source_stats)
print(f"\nTotal with source: {df_all['has_source'].sum()} / {len(df_all)} ({df_all['has_source'].mean()*100:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 4))
labels = source_stats.index
has = source_stats["with_source"]
missing = source_stats["total"] - source_stats["with_source"]
ax.bar(labels, has, label="Has source code", color="#2ecc71", edgecolor="white")
ax.bar(labels, missing, bottom=has, label="Metadata only", color="#95a5a6", edgecolor="white")
ax.set_ylabel("Count")
ax.set_title("Source Code Availability", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## Key Observations

1. **Severe class imbalance** — the dataset is ~14:1 benign-to-malicious.
   This must be addressed during model training (via `scale_pos_weight` in XGBoost).

2. **Both registries represented** — PyPI and npm packages appear in both classes,
   ensuring the model generalizes across ecosystems.

3. **Missing fields are common** — `version`, `source_url`, and `source_code` are
   frequently absent. Layer 1 (metadata) is the most reliable signal in this dataset.

4. **Name length differs subtly** — malicious packages tend to have names close in
   length to popular packages (a hallmark of typosquatting).

5. **Source code is sparse** — most records are metadata-only. In the live pipeline,
   Layers 2–3 become active when source tarballs are fetched on demand.

---
*Next: [02_feature_engineering.ipynb](02_feature_engineering.ipynb) — transform these
records into the 7-dimensional feature vector used by the classifier.*